# Gene Set Enrichment Test Dataset

Builds `test-data/habib17-enrichment-test-data-format.zarr` by extending the
differential-expression (DE) test store with a universal **gene set enrichment**
schema computed with [decoupler](https://decoupler-py.readthedocs.io/) 2.x.

The new store is self-contained: it is a verbatim copy of the DE store (`X`,
`obs`, `var`, `uns/de`) plus a new `uns/enrichment` group (contrast-based
enrichment, mirroring the DE folder/registry pattern) and per-cell activity
matrices in `obsm/`.

Run the DE notebook (`calculate-differential-expression-data.ipynb`) first;
this notebook reuses its output verbatim. Requires network access for
decoupler's `dc.op` gene set downloads (OmniPath / MSigDB).

In [ ]:
from pathlib import Path
import json
import shutil
from datetime import datetime, timezone
from dataclasses import dataclass, field
from typing import Optional
import importlib.metadata

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
import zarr
from zarr.core.dtype import VariableLengthUTF8
import decoupler as dc

np.random.seed(1)

## Configuration

In [ ]:
# Reuse the existing DE store verbatim as the base for the enrichment store.
DE_STORE = Path("test-data/habib17-differential-expression-test-data-format.zarr")
OUTPUT_PATH = Path("test-data/habib17-enrichment-test-data-format.zarr")

ORGANISM = "human"

# All 15 one-vs-rest wilcoxon contrasts (de_001..de_015) carry a signed effect
# size + scores, so they are the signature source for enrichment.
FULL_CONTRASTS = [f"de_{i:03d}" for i in range(1, 16)]
# A shared 5-contrast subset keeps the large collections comparable and bounded.
SUBSET_CONTRASTS = [f"de_{i:03d}" for i in range(1, 6)]

# Number of top up-regulated genes ORA treats as the observed signature.
ORA_N_UP = 50

# zero/underflowed p -> cap = ceil(max_finite * this) for the significance array.
SIGNIFICANCE_GAP_FACTOR = 1.5

# Minimum genes per set that must be present in the data (decoupler `tmin`).
TMIN = 5

## Gene set collections (decoupler `dc.op`)

Each collection is a long-format `net` (`source`, `target`, optional `weight`).
Downloads are wrapped so a single unreachable resource is skipped with a log
line rather than aborting the whole run.

In [ ]:
def _load_gobp() -> pd.DataFrame:
    """MSigDB C5 GO:Biological Process subcollection as an unweighted net."""
    msig = dc.op.resource("MSigDB", organism=ORGANISM)
    gobp = msig[msig["collection"] == "go_biological_process"]
    return gobp.rename(columns={"geneset": "source", "genesymbol": "target"})[
        ["source", "target"]
    ]


def _load_omnipath() -> pd.DataFrame:
    """OmniPath annotations (Wang subcellular locations) as unweighted gene sets."""
    wang = dc.op.resource("Wang", organism=ORGANISM)
    net = wang.dropna(subset=["location"]).rename(
        columns={"location": "source", "genesymbol": "target"}
    )[["source", "target"]]
    return net[net["source"].astype(str).str.len() > 0]


def load_collections() -> dict:
    """Download every gene set collection, skipping any that fail."""
    collections: dict = {}

    def add(key: str, label: str, loader) -> None:
        try:
            net = loader().dropna(subset=["source", "target"]).copy()
            net["source"] = net["source"].astype(str)
            net["target"] = net["target"].astype(str)
            net = net.drop_duplicates(subset=["source", "target"])
            collections[key] = {
                "net": net,
                "label": label,
                "weighted": "weight" in net.columns,
                "set_size": net.groupby("source")["target"].nunique(),
            }
            print(f"  ✓ {label}: {net['source'].nunique()} sets, {len(net)} edges")
        except Exception as exc:  # noqa: BLE001 - want to skip, not abort
            print(f"  ✗ {label}: {exc}")

    add("hallmark", "MSigDB Hallmark",
        lambda: dc.op.hallmark(organism=ORGANISM)[["source", "target"]])
    add("gobp", "GO:BP", _load_gobp)
    add("omnipath", "OmniPath", _load_omnipath)
    add("progeny", "PROGENy",
        lambda: dc.op.progeny(organism=ORGANISM)[["source", "target", "weight"]])
    add("collectri", "CollecTRI",
        lambda: dc.op.collectri(organism=ORGANISM)[["source", "target", "weight"]])
    return collections

## Signature matrix from `uns/de`

Rows are DE contrasts, columns are all `var_names`, values are the per-gene DE
`scores` (reindexed to the full gene axis; genes absent from a contrast filled
with 0).

In [ ]:
def load_de_scores(store_path: Path, contrast_ids: list[str], var_names: np.ndarray) -> pd.DataFrame:
    """Assemble a contrasts x genes signature matrix from DE score arrays."""
    # use_consolidated=False: the manually written de_XXX folders are not in the
    # store's consolidated metadata index. The uns/de directory also has no group
    # marker, so index each contrast folder by full path.
    root = zarr.open_group(str(store_path), mode="r", use_consolidated=False)
    rows: dict[str, pd.Series] = {}
    for cid in contrast_ids:
        grp = root[f"uns/de/{cid}"]
        # gene_id reads back as numpy StringDType; pd.Index handles the cast.
        gene_id = pd.Index(grp["gene_id"][:]).astype(str)
        scores = np.asarray(grp["scores"][:], dtype=float)
        series = pd.Series(scores, index=gene_id)
        series = series[~series.index.duplicated(keep="first")]
        rows[cid] = series.reindex(var_names).fillna(0.0)
    mat = pd.DataFrame(rows).T
    mat.columns = var_names
    return mat

## Contrast-based enrichment → `uns/enrichment/es_XXX/`

The method × collection matrix. Each (contrast, method, collection) triple is
one `es_XXX` folder. `signed` controls whether a signed effect size is exposed
(GSEA/MLM/ULM yes; ORA reports an over-representation statistic only).

In [ ]:
@dataclass
class EnrichJob:
    method: str            # decoupler dc.mt.* method name
    collection: str        # collection key
    contrasts: list[str]   # DE contrast ids to run
    signed: bool           # whether a signed effect size is exposed
    effect_size_label: Optional[str]
    input_statistic: str


def build_jobs() -> list[EnrichJob]:
    return [
        EnrichJob("gsea", "hallmark", FULL_CONTRASTS, True,
                  "Normalized Enrichment Score", "de_scores"),
        EnrichJob("ora", "hallmark", FULL_CONTRASTS, False,
                  None, f"top_{ORA_N_UP}_up_genes"),
        EnrichJob("gsea", "gobp", SUBSET_CONTRASTS, True,
                  "Normalized Enrichment Score", "de_scores"),
        EnrichJob("ora", "gobp", SUBSET_CONTRASTS, False,
                  None, f"top_{ORA_N_UP}_up_genes"),
        EnrichJob("ora", "omnipath", SUBSET_CONTRASTS, False,
                  None, f"top_{ORA_N_UP}_up_genes"),
        EnrichJob("mlm", "progeny", SUBSET_CONTRASTS, True,
                  "MLM t-value", "de_scores"),
        EnrichJob("ulm", "collectri", SUBSET_CONTRASTS, True,
                  "ULM t-value", "de_scores"),
    ]


def run_method(method: str, mat: pd.DataFrame, net: pd.DataFrame):
    """Run a decoupler method; returns (score_df, padj_df) shaped contrasts x sources."""
    fn = getattr(dc.mt, method)
    kwargs = {"tmin": TMIN}
    if method == "ora":
        kwargs["n_up"] = ORA_N_UP
    out = fn(mat, net, verbose=False, **kwargs)
    score, padj = out if isinstance(out, tuple) else (out, None)
    return score, padj


def overlap_sizes(net: pd.DataFrame, var_names: np.ndarray) -> pd.Series:
    """Genes per set that are present/tested in the data."""
    present = net[net["target"].isin(set(map(str, var_names)))]
    return present.groupby("source")["target"].nunique()

## Zarr writer helpers (shared conventions with the DE notebook)

In [ ]:
def _write_array(group: zarr.Group, field_name: str, arr: np.ndarray) -> None:
    """Write a 1D array; strings use VariableLengthUTF8 for cross-library reads."""
    if arr.dtype.kind in {"O", "U"}:
        string_array = group.create_array(
            field_name,
            shape=arr.shape,
            dtype=VariableLengthUTF8(),
            fill_value="",
        )
        string_array[:] = np.asarray(arr, dtype=object)
        return
    group[field_name] = arr


def _presort_enrichment_arrays(arrays: dict) -> dict:
    """Sort every array descending by scores (NaN last), stable tie-breakers."""
    scores = arrays["scores"]
    pvals_adj = arrays["pvals_adj"]
    gene_set_id = arrays["gene_set_id"]
    sort_idx = np.lexsort((
        gene_set_id,                                          # ascending id
        np.where(np.isnan(pvals_adj), np.inf, pvals_adj),    # ascending padj
        -np.where(np.isnan(scores), -np.inf, scores),        # descending scores
    ))
    return {key: arr[sort_idx] for key, arr in arrays.items()}


def _compute_significance(
    pvals_adj: np.ndarray,
    gap_factor: float = SIGNIFICANCE_GAP_FACTOR,
) -> tuple[np.ndarray, Optional[float]]:
    """Plot-ready -log10(adjusted p) with a data-driven cap.

    Returns (significance_array, cap). NaN inputs stay NaN. Zero/underflowed
    p-values (infinite -log10) are pinned to cap = ceil(max_finite * gap_factor),
    leaving a readable gap above the largest real value; the cap always equals
    np.nanmax(significance).
    """
    padj = np.asarray(pvals_adj, dtype=float)
    significance = np.full(padj.shape, np.nan)
    valid = ~np.isnan(padj)
    if not np.any(valid):
        return significance, None
    with np.errstate(divide="ignore"):
        logp = -np.log10(padj[valid])  # padj == 0 -> inf
    finite = logp[np.isfinite(logp)]
    if len(finite) == 0:
        cap = 300.0  # degenerate: every valid p underflowed to 0
    elif np.any(np.isinf(logp)):
        cap = float(np.ceil(np.max(finite) * gap_factor))
    else:
        cap = float(np.ceil(np.max(finite)))
    significance[valid] = np.minimum(np.where(np.isinf(logp), cap, logp), cap)
    return significance, cap


def _compute_axis_bounds(effect_size: np.ndarray, pvals_adj: np.ndarray):
    """effect_size_max (abs, ceil) and significance_max (-log10, capped 300)."""
    valid_effect = effect_size[~(np.isnan(effect_size) | np.isinf(effect_size))]
    valid_pvals = pvals_adj[~np.isnan(pvals_adj)]
    effect_size_max = None
    significance_max = None
    if len(valid_effect) > 0:
        effect_size_max = float(np.ceil(np.max(np.abs(valid_effect))))
    if len(valid_pvals) > 0:
        logp = -np.log10(np.maximum(valid_pvals, 1e-300))
        significance_max = min(300.0, float(np.ceil(np.max(logp))))
    return effect_size_max, significance_max


def assemble_arrays(cid, score_df, padj_df, set_size, ov, signed) -> dict:
    """Build the stable per-folder array set for one contrast row."""
    sources = np.asarray(score_df.columns).astype(str)
    scores = np.asarray(score_df.loc[cid].to_numpy(), dtype=float)
    if padj_df is not None:
        pvals_adj = np.asarray(padj_df.loc[cid].to_numpy(), dtype=float)
    else:
        pvals_adj = np.full(len(sources), np.nan)
    effect_size = scores.copy() if signed else np.full(len(sources), np.nan)
    set_size_arr = set_size.reindex(sources).to_numpy(dtype=float)
    overlap_arr = ov.reindex(sources).fillna(0).to_numpy(dtype=float)
    return {
        "gene_set_id": sources,
        "scores": scores,
        "effect_size": effect_size,
        "pvals": np.full(len(sources), np.nan),  # decoupler returns adjusted only
        "pvals_adj": pvals_adj,
        "significance": _compute_significance(pvals_adj)[0],
        "set_size": set_size_arr,
        "overlap_size": overlap_arr,
    }


def write_enrichment_folder(zarr_path, es_id, cid, arrays, job, collection_label, de_meta) -> dict:
    """Write one es_XXX folder and return its registry entry."""
    folder = zarr_path / "uns" / "enrichment" / es_id
    folder.mkdir(parents=True, exist_ok=True)
    store = zarr.open(str(folder), mode="w")
    for field_name, arr in arrays.items():
        _write_array(store, field_name, arr)

    effect_size_max, _ = _compute_axis_bounds(
        arrays["effect_size"], arrays["pvals_adj"]
    )
    # significance_max is the plot cap, sourced from the precomputed array.
    significance_arr = arrays["significance"]
    significance_max = (
        float(np.nanmax(significance_arr))
        if np.any(~np.isnan(significance_arr))
        else None
    )
    has_effect_size = bool(job.signed and effect_size_max is not None)
    has_significance = significance_max is not None
    has_set_size = bool(np.any(~np.isnan(arrays["set_size"])))
    has_overlap = bool(np.any(~np.isnan(arrays["overlap_size"])))

    return {
        "enrichment_id": es_id,
        "source_contrast_id": cid,
        "group_1": de_meta["group_1"],
        "group_2": de_meta["group_2"],
        "test_type": de_meta["test_type"],
        "subset_column": de_meta["subset_column"],
        "subset_value": de_meta["subset_value"],
        "contrast_column": de_meta["contrast_column"],
        "enrichment_method": job.method,
        "gene_set_collection": collection_label,
        "n_gene_sets": int(len(arrays["gene_set_id"])),
        "input_statistic": job.input_statistic,
        "feature_type": "Gene Set",
        "has_effect_size": has_effect_size,
        "has_significance": has_significance,
        "has_set_size": has_set_size,
        "has_overlap": has_overlap,
        "has_leading_edge": False,
        "correction_method": "benjamini-hochberg" if has_significance else None,
        "effect_size_label": job.effect_size_label if has_effect_size else None,
        "significance_label": "-log10(FDR)" if has_significance else None,
        "effect_size_max": effect_size_max if has_effect_size else None,
        "significance_max": significance_max if has_significance else None,
        "top_10_gene_set_ids": arrays["gene_set_id"].tolist()[:10],
    }

## Per-cell activity → `obsm/`

A few methods run on the full expression matrix, producing cells × gene sets
activity matrices. Each is written as an `obsm/X_activity_<method>_<collection>`
array (column order recorded as `gene_set_ids` in the registry).

In [ ]:
CELL_ACTIVITY_COMBOS = [
    ("mlm", "progeny", "MLM t-value"),
    ("ulm", "collectri", "ULM t-value"),
    ("aucell", "hallmark", "AUCell enrichment score"),
]


def run_cell_activities(adata, collections) -> list[dict]:
    """Compute per-cell activity matrices; returns metadata + score DataFrames."""
    X = adata.X
    dense = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
    expr = pd.DataFrame(
        dense,
        index=adata.obs_names.astype(str),
        columns=adata.var_names.astype(str),
    )
    activities: list[dict] = []
    for method, key, label in CELL_ACTIVITY_COMBOS:
        if key not in collections:
            print(f"  ✗ activity {method} x {key}: collection unavailable")
            continue
        net = collections[key]["net"]
        score, _padj = run_method(method, expr, net)
        obsm_key = f"X_activity_{method}_{key}"
        activities.append({
            "obsm_key": obsm_key,
            "method": method,
            "collection_key": key,
            "gene_set_collection": collections[key]["label"],
            "value_label": label,
            "gene_set_ids": [str(c) for c in score.columns],
            "score": score,
        })
        print(f"  ✓ activity {obsm_key}: {score.shape[0]} cells x {score.shape[1]} sets")
    return activities


def write_obsm_array(zarr_path: Path, key: str, matrix: np.ndarray) -> None:
    """Append a dense obsm activity matrix with AnnData array encoding attrs."""
    obsm = zarr.open_group(str(zarr_path / "obsm"), mode="a")
    matrix = np.asarray(matrix, dtype="float32")
    arr = obsm.create_array(
        key,
        shape=matrix.shape,
        chunks=matrix.shape,
        dtype="float32",
        fill_value=0.0,
        overwrite=True,
    )
    arr[:] = matrix
    arr.attrs["encoding-type"] = "array"
    arr.attrs["encoding-version"] = "0.2.0"

## Finalize: consolidated metadata

The DE folders were copied and the enrichment folders / activity matrices are
written manually, so they carry no AnnData encoding attributes and are absent
from the store's consolidated metadata index. This step tags every
manually-written node with the right `encoding-type`, adds the `uns/de` and
`uns/enrichment` group markers, and rebuilds the root consolidated metadata so
the whole store is self-describing and reads back cleanly with `anndata`.


In [ ]:
def _is_string_array(arr) -> bool:
    """Whether a zarr array holds strings (VariableLengthUTF8)."""
    if getattr(arr.dtype, "kind", "") in ("U", "S", "O", "T"):
        return True
    return "string" in str(getattr(arr.metadata, "data_type", "")).lower()


def _set_encoding(node, encoding_type: str, encoding_version: str) -> None:
    node.attrs["encoding-type"] = encoding_type
    node.attrs["encoding-version"] = encoding_version


def finalize_metadata(store_path: Path) -> None:
    """Tag manually-written nodes with AnnData encoding + rebuild consolidated metadata."""
    store_path = Path(store_path)
    # AnnData encoding attrs so anndata can read uns/de and uns/enrichment as
    # nested dicts of arrays (dict groups; string vs numeric arrays).
    for sub in ["uns/de", "uns/enrichment"]:
        sub_dir = store_path / sub
        if not sub_dir.exists():
            continue
        parent = zarr.open_group(str(sub_dir), mode="a")  # creates the group marker if absent
        _set_encoding(parent, "dict", "0.1.0")
        for child_dir in sorted(p for p in sub_dir.iterdir() if p.is_dir()):
            child = zarr.open_group(str(child_dir), mode="a")
            _set_encoding(child, "dict", "0.1.0")
            for arr_name in list(child.array_keys()):
                arr = child[arr_name]
                _set_encoding(arr, "string-array" if _is_string_array(arr) else "array", "0.2.0")
    # Rebuild the root consolidated metadata over the full hierarchy.
    root = zarr.open_group(str(store_path), mode="a", use_consolidated=False)
    zarr.consolidate_metadata(root.store)
    print("Refreshed consolidated metadata (all uns/de, uns/enrichment and obsm nodes indexed)")

## Build the store

In [ ]:
def main() -> None:
    if not DE_STORE.exists():
        raise FileNotFoundError(
            f"DE store not found: {DE_STORE}. Run "
            "calculate-differential-expression-data.ipynb first."
        )

    # 1. Self-contained base: copy the DE store verbatim (X, obs, var, uns/de).
    if OUTPUT_PATH.exists():
        shutil.rmtree(OUTPUT_PATH)
    shutil.copytree(DE_STORE, OUTPUT_PATH)
    print(f"Copied DE store → {OUTPUT_PATH}")

    adata = ad.read_zarr(OUTPUT_PATH)
    var_names = adata.var_names.astype(str).to_numpy()
    print(f"Base AnnData: {adata.n_obs} cells x {adata.n_vars} genes")

    de_registry = json.load(open(OUTPUT_PATH / "uns" / "de" / "contrast_registry.json"))
    de_by_id = {c["contrast_id"]: c for c in de_registry["contrasts"]}

    # 2. Gene set collections.
    print("Downloading gene set collections...")
    collections = load_collections()

    # 3. Contrast-based enrichment.
    signature = load_de_scores(OUTPUT_PATH, FULL_CONTRASTS, var_names)
    jobs = build_jobs()
    registry: list[dict] = []
    es_counter = 1
    print("Running contrast-based enrichment...")
    for job in jobs:
        if job.collection not in collections:
            print(f"  ✗ {job.method} x {job.collection}: collection unavailable, skipped")
            continue
        collection = collections[job.collection]
        contrasts = [c for c in job.contrasts if c in de_by_id and c in signature.index]
        try:
            score_df, padj_df = run_method(job.method, signature.loc[contrasts], collection["net"])
        except Exception as exc:  # noqa: BLE001
            print(f"  ✗ {job.method} x {job.collection}: {exc}")
            continue
        set_size = collection["set_size"]
        ov = overlap_sizes(collection["net"], var_names)
        for cid in contrasts:
            if cid not in score_df.index:
                print(f"    ⊘ {cid}: no result for {job.method} x {job.collection}")
                continue
            es_id = f"es_{es_counter:03d}"
            arrays = assemble_arrays(cid, score_df, padj_df, set_size, ov, job.signed)
            arrays = _presort_enrichment_arrays(arrays)
            meta = write_enrichment_folder(
                OUTPUT_PATH, es_id, cid, arrays, job, collection["label"], de_by_id[cid]
            )
            registry.append(meta)
            es_counter += 1
        print(f"  ✓ {job.method} x {collection['label']}: {len(contrasts)} contrasts")

    # 4. Per-cell activity matrices.
    print("Running per-cell activities...")
    activities = run_cell_activities(adata, collections)
    cell_activities: list[dict] = []
    for act in activities:
        write_obsm_array(OUTPUT_PATH, act["obsm_key"], act["score"].to_numpy())
        cell_activities.append({
            "obsm_key": act["obsm_key"],
            "method": act["method"],
            "gene_set_collection": act["gene_set_collection"],
            "value_label": act["value_label"],
            "gene_set_ids": act["gene_set_ids"],
            "n_gene_sets": len(act["gene_set_ids"]),
        })

    # 5. Registry.
    enrichment_summary = {
        "total_enrichments": len(registry),
        "total_cell_activities": len(cell_activities),
        "series": [
            {
                "method": job.method,
                "gene_set_collection": collections[job.collection]["label"]
                if job.collection in collections else job.collection,
                "n_contrasts": len([c for c in job.contrasts if c in de_by_id]),
                "available": job.collection in collections,
            }
            for job in jobs
        ],
    }
    registry_json = {
        "version": "1.0",
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "decoupler_version": importlib.metadata.version("decoupler"),
        "zarr_version": importlib.metadata.version("zarr"),
        "dataset_metadata": {
            "n_obs": int(adata.n_obs),
            "n_vars": int(adata.n_vars),
            "source_store": DE_STORE.name,
        },
        "enrichment_summary": enrichment_summary,
        "contrast_enrichments": registry,
        "cell_activities": cell_activities,
    }
    registry_path = OUTPUT_PATH / "uns" / "enrichment" / "enrichment_registry.json"
    registry_path.parent.mkdir(parents=True, exist_ok=True)
    with open(registry_path, "w") as fh:
        json.dump(registry_json, fh, indent=2)

    # 6. Refresh consolidated metadata so every written node is indexed.
    finalize_metadata(OUTPUT_PATH)

    print(f"\nWrote {len(registry)} enrichment folders + {len(cell_activities)} "
          f"cell-activity matrices to {OUTPUT_PATH}")
    print(f"Registry: {registry_path}")

In [ ]:
if __name__ == "__main__":
    main()